In [1]:
from google.colab import drive
drive.mount('/content/drive')

ROOT = '/content/drive/MyDrive/capstone-ml'
DATA = f'{ROOT}/data'
PRED = f'{ROOT}/predictions'
MODELS = f'{ROOT}/models'
LOGS = f'{ROOT}/logs'

import sys, importlib.metadata as md
import tensorflow as tf
print('Python', sys.version.split()[0])
for p in ['tensorflow', 'keras', 'numpy', 'pandas', 'scikit-learn']:
    print(f'{p}=={md.version(p)}')
print('GPU:', tf.config.list_physical_devices('GPU'))

Mounted at /content/drive
Python 3.13.16
tensorflow==2.21.0
keras==3.13.2
numpy==2.4.6
pandas==2.2.3
scikit-learn==1.6.1
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
import json, os
import pandas as pd

splits = {s: pd.read_csv(f'{DATA}/sst5_{s}.csv') for s in ['train', 'val', 'test']}
assert splits['train']['label'].value_counts().sort_index().tolist() == [3310, 1624, 3610]

with open(f'{DATA}/class_weights.json') as f:
    CLASS_WEIGHTS = {int(k): v for k, v in json.load(f)['weights'].items()}
print({s: len(df) for s, df in splits.items()}, CLASS_WEIGHTS)

{'train': 8544, 'val': 1101, 'test': 2210} {0: 0.8604, 1: 1.7537, 2: 0.7889}


In [3]:
import re

def glove_tokens(text):
    t = re.sub(r"n't\b", " n't", text)                  # doesn't -> does n't
    t = re.sub(r"'(s|re|ve|ll|d|m)\b", r" '\1", t)       # it's -> it 's
    t = re.sub(r'([.,!?;:()"])', r' \1 ', t)            # punctuation as its own token
    return t.split()

for t in splits['train']['text'].head(3):
    print(t)
    print('  ->', glove_tokens(t))

a stirring, funny and finally transporting re-imagining of beauty and the beast and 1930s horror films
  -> ['a', 'stirring', ',', 'funny', 'and', 'finally', 'transporting', 're-imagining', 'of', 'beauty', 'and', 'the', 'beast', 'and', '1930s', 'horror', 'films']
apparently reassembled from the cutting-room floor of any given daytime soap.
  -> ['apparently', 'reassembled', 'from', 'the', 'cutting-room', 'floor', 'of', 'any', 'given', 'daytime', 'soap', '.']
they presume their audience won't sit still for a sociology lesson, however entertainingly presented, so they trot out the conventional science-fiction elements of bug-eyed monsters and futuristic women in skimpy clothes.
  -> ['they', 'presume', 'their', 'audience', 'wo', "n't", 'sit', 'still', 'for', 'a', 'sociology', 'lesson', ',', 'however', 'entertainingly', 'presented', ',', 'so', 'they', 'trot', 'out', 'the', 'conventional', 'science-fiction', 'elements', 'of', 'bug-eyed', 'monsters', 'and', 'futuristic', 'women', 'in', 'ski

In [4]:
import numpy as np
from collections import Counter

glove = {}
with open(f'{DATA}/glove.6B.100d.txt', encoding='utf8') as f:
    for line in f:
        word, *vec = line.rstrip().split(' ')
        glove[word] = np.asarray(vec, dtype='float32')
print(f'GloVe vectors loaded: {len(glove):,}')

def coverage(token_lists):
    c = Counter(w for sent in token_lists for w in sent)
    types = sum(w in glove for w in c) / len(c)
    tokens = sum(n for w, n in c.items() if w in glove) / sum(c.values())
    return len(c), types, tokens, c

for name, fn in [('plain split', str.split), ('GloVe-style', glove_tokens)]:
    n, ty, to, _ = coverage([fn(t) for t in splits['train']['text']])
    print(f'{name:12s}: {n:,} distinct words, {ty:.1%} have a GloVe vector, covering {to:.1%} of all word uses')

tokens = {s: [glove_tokens(t) for t in df['text']] for s, df in splits.items()}
_, _, _, train_counts = coverage(tokens['train'])
missing = [(w, n) for w, n in train_counts.most_common() if w not in glove]
print('\nMost frequent training words with no GloVe vector:', missing[:15])

lens = np.array([len(s) for s in tokens['train']])
print(f'\nTokens per sentence: median {int(np.median(lens))}, longest {lens.max()}, over 64: {(lens > 64).mean():.2%}')

GloVe vectors loaded: 400,000
plain split : 23,715 distinct words, 56.4% have a GloVe vector, covering 84.9% of all word uses
GloVe-style : 16,532 distinct words, 92.3% have a GloVe vector, covering 99.1% of all word uses

Most frequent training words with no GloVe vector: [('cletis', 9), ('feardotcom', 6), ('waydowntown', 5), ('birot', 5), ('ozpetek', 5), ('wollter', 4), ('punch-drunk', 4), ('kosashvili', 4), ('dumbed-down', 4), ('mother/daughter', 3), ('bodice-ripper', 3), ('cliche-ridden', 3), ('seldahl', 3), ('janklowicz-mann', 3), ('ham-fisted', 3)]

Tokens per sentence: median 19, longest 53, over 64: 0.00%


In [5]:
service_words = ['plumber', 'electrician', 'fundi', 'punctual', 'overcharged', 'tidy', 'leaking',
                 'rewiring', 'invoice', 'unreliable', 'rescheduled', 'courteous', 'quote', 'workmanship']
for w in service_words:
    print(f'{w:12s}  in SST-5 training: {w in train_counts!s:5s}  has GloVe vector: {w in glove}')

plumber       in SST-5 training: False  has GloVe vector: True
electrician   in SST-5 training: False  has GloVe vector: True
fundi         in SST-5 training: False  has GloVe vector: True
punctual      in SST-5 training: False  has GloVe vector: True
overcharged   in SST-5 training: False  has GloVe vector: True
tidy          in SST-5 training: False  has GloVe vector: True
leaking       in SST-5 training: False  has GloVe vector: True
rewiring      in SST-5 training: False  has GloVe vector: True
invoice       in SST-5 training: False  has GloVe vector: True
unreliable    in SST-5 training: False  has GloVe vector: True
rescheduled   in SST-5 training: False  has GloVe vector: True
courteous     in SST-5 training: False  has GloVe vector: True
quote         in SST-5 training: False  has GloVe vector: True
workmanship   in SST-5 training: False  has GloVe vector: True


In [6]:
from keras.utils import pad_sequences

MAX_LEN = 64
GLOVE_EXTRA = 100_000
EMB_DIM = 100

top_glove = list(glove)[:GLOVE_EXTRA]                                     # GloVe file is ordered by frequency
train_words = [w for w, _ in train_counts.most_common() if w in glove]
vocab = list(dict.fromkeys(train_words + top_glove))                     # training words first, no duplicates
word_index = {w: i + 2 for i, w in enumerate(vocab)}                     # 0 = padding, 1 = unknown

rng = np.random.default_rng(42)
std = float(np.stack([glove[w] for w in top_glove]).std())
emb_matrix = np.zeros((len(vocab) + 2, EMB_DIM), dtype='float32')
emb_matrix[1] = rng.normal(0, std, EMB_DIM)
for w, i in word_index.items():
    emb_matrix[i] = glove[w]

def to_ids(tokens):
    return [word_index.get(w, 1) for w in tokens]

seqs = {s: pad_sequences([to_ids(t) for t in tokens[s]], maxlen=MAX_LEN, padding='post', truncating='pre')
        for s in tokens}
labels = {s: splits[s]['label'].values for s in splits}

print(f'Vocabulary: {len(vocab):,} words ({len(train_words):,} from training, the rest from GloVe)')
print(f'Embedding matrix: {emb_matrix.shape}, {emb_matrix.nbytes / 1e6:.0f} MB')
for s in ['train', 'val', 'test']:
    ids = np.concatenate([to_ids(t) for t in tokens[s]])
    print(f'{s}: {seqs[s].shape}, {(ids == 1).mean():.2%} of tokens map to unknown')

Vocabulary: 101,583 words (15,252 from training, the rest from GloVe)
Embedding matrix: (101585, 100), 41 MB
train: (8544, 64), 0.86% of tokens map to unknown
val: (1101, 64), 1.62% of tokens map to unknown
test: (2210, 64), 1.71% of tokens map to unknown


In [7]:
import keras
from keras import layers

def build_model():
    inp = keras.Input(shape=(MAX_LEN,), dtype='int32')
    x = layers.Embedding(emb_matrix.shape[0], EMB_DIM, mask_zero=True, trainable=False, name='glove')(inp)
    x = layers.Bidirectional(layers.LSTM(64))(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(3, activation='softmax')(x)
    model = keras.Model(inp, out)
    model.get_layer('glove').set_weights([emb_matrix])
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

build_model().summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 64)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ glove (Embedding)   │ (None, 64, 100)   │ 10,158,500 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 64)        │          0 │ input_layer[0][0] │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 128)       │     84,480 │ glove[0][0],      │
│ (Bidirectional)     │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128)       │          0 │ bidirectional[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 3)         │        387 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 10,243,367 (39.08 MB)

 Trainable params: 84,867 (331.51 KB)

 Non-trainable params: 10,158,500 (38.75 MB)

In [8]:
import time
from datetime import date
from sklearn.metrics import f1_score, accuracy_score

NAMES = ['negative', 'neutral', 'positive']
os.makedirs(PRED, exist_ok=True)
os.makedirs(MODELS, exist_ok=True)
log = f'{LOGS}/experiments.csv'

def log_run(row):
    logged = pd.read_csv(log)
    logged = logged[logged['run_id'].astype(str) != row['run_id']]
    pd.concat([logged, pd.DataFrame([row])]).to_csv(log, index=False)

def prediction_frame(split, proba, seed, eval_set):
    df = splits[split]
    return pd.DataFrame({
        'id': df['id'], 'true_label': df['label'], 'pred_label': proba.argmax(axis=1),
        'p_neg': proba[:, 0].round(6), 'p_neu': proba[:, 1].round(6), 'p_pos': proba[:, 2].round(6),
        'model': 'lstm', 'seed': seed, 'train_data': 'sst5', 'eval_set': eval_set,
    })

results = []
for seed in [42, 43, 44]:
    keras.utils.set_random_seed(seed)
    model = build_model()
    stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    start = time.perf_counter()
    hist = model.fit(seqs['train'], labels['train'], validation_data=(seqs['val'], labels['val']),
                     epochs=30, batch_size=32, class_weight=CLASS_WEIGHTS, callbacks=[stop], verbose=0)
    best_epoch = int(np.argmin(hist.history['val_loss'])) + 1

    scores = {}
    for s in ['val', 'test']:
        proba = model.predict(seqs[s], verbose=0)
        prediction_frame(s, proba, seed, f'sst5_{s}').to_csv(f'{PRED}/lstm_seed{seed}_sst5_{s}.csv', index=False)
        scores[s] = f1_score(labels[s], proba.argmax(axis=1), average='macro')
    model.save(f'{MODELS}/lstm_seed{seed}.keras')

    log_run({'run_id': f'lstm-sst5-seed{seed}', 'model': 'lstm', 'seed': seed, 'train_data': 'sst5',
             'hyperparameters': f'glove 6B 100d frozen; vocab train words with glove + top 100k glove; maxlen {MAX_LEN}; '
                                f'bilstm 64; dropout 0.5; adam 1e-3; batch 32; early stop val_loss patience 3 (best epoch {best_epoch})',
             'val_macro_f1': round(scores['val'], 4), 'test_macro_f1': round(scores['test'], 4),
             'date': date.today().isoformat()})
    results.append({'seed': seed, 'best_epoch': best_epoch, 'epochs_run': len(hist.history['val_loss']),
                    'val_macro_f1': round(scores['val'], 4), 'test_macro_f1': round(scores['test'], 4),
                    'minutes': round((time.perf_counter() - start) / 60, 1)})
    print(results[-1])

res = pd.DataFrame(results)
print(f"\nValidation macro-F1: mean {res['val_macro_f1'].mean():.4f}, std {res['val_macro_f1'].std():.4f}")
print(f"Test macro-F1:       mean {res['test_macro_f1'].mean():.4f}, std {res['test_macro_f1'].std():.4f}")

{'seed': 42, 'best_epoch': 6, 'epochs_run': 9, 'val_macro_f1': 0.537, 'test_macro_f1': 0.5669, 'minutes': 0.7}
{'seed': 43, 'best_epoch': 5, 'epochs_run': 8, 'val_macro_f1': 0.5492, 'test_macro_f1': 0.5718, 'minutes': 0.5}
{'seed': 44, 'best_epoch': 6, 'epochs_run': 9, 'val_macro_f1': 0.5766, 'test_macro_f1': 0.5748, 'minutes': 0.5}

Validation macro-F1: mean 0.5543, std 0.0203
Test macro-F1:       mean 0.5712, std 0.0040


In [9]:
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix

runs = [('svm', '', f'{PRED}/svm_sst5_test.csv')] + \
       [('lstm', s, f'{PRED}/lstm_seed{s}_sst5_test.csv') for s in [42, 43, 44]]

rows = []
for model_name, seed, path in runs:
    p = pd.read_csv(path)
    prec, rec, f1, _ = precision_recall_fscore_support(p['true_label'], p['pred_label'], labels=[0, 1, 2], zero_division=0)
    rows.append({'model': model_name, 'seed': seed,
                 'neg_f1': round(f1[0], 4), 'neu_recall': round(rec[1], 4), 'neu_precision': round(prec[1], 4),
                 'neu_f1': round(f1[1], 4), 'pos_f1': round(f1[2], 4), 'macro_f1': round(f1.mean(), 4)})
table = pd.DataFrame(rows)
print(table.to_string(index=False))

mid_seed = int(res.iloc[(res['test_macro_f1'] - res['test_macro_f1'].mean()).abs().argmin()]['seed'])
for name, path in [('SVM', f'{PRED}/svm_sst5_test.csv'), (f'LSTM seed {mid_seed}', f'{PRED}/lstm_seed{mid_seed}_sst5_test.csv')]:
    p = pd.read_csv(path)
    print(f'\n{name}, test confusion matrix (rows = true, columns = predicted):')
    print(pd.DataFrame(confusion_matrix(p['true_label'], p['pred_label']),
                       index=['true neg', 'true neu', 'true pos'], columns=['pred neg', 'pred neu', 'pred pos']))

model seed  neg_f1  neu_recall  neu_precision  neu_f1  pos_f1  macro_f1
  svm       0.6920      0.1414         0.2880  0.1897  0.7382    0.5399
 lstm   42  0.7324      0.1645         0.3832  0.2302  0.7380    0.5669
 lstm   43  0.6373      0.4473         0.2727  0.3389  0.7392    0.5718
 lstm   44  0.6341      0.4807         0.2812  0.3548  0.7355    0.5748

SVM, test confusion matrix (rows = true, columns = predicted):
          pred neg  pred neu  pred pos
true neg       664        87       161
true neu       192        55       142
true pos       151        49       709

LSTM seed 43, test confusion matrix (rows = true, columns = predicted):
          pred neg  pred neu  pred pos
true neg       507       290       115
true neu       103       174       112
true pos        69       174       666
